# Week 1 — Fundamentals

This week has two halves.

The **lecture** is about the process: what a statistical question looks like,
what a variable is, what a sample is for, and how to describe data honestly.

This **notebook** is the practical half. By the end of it you will be able to
load a dataset, find out what is in it, and describe it — which is the first
thing you will do in every week that follows.

Nothing here assumes you have used Python for data before.

---

## Part 1 — The tools

Three pieces:

| Tool | What it is for |
|---|---|
| **Python** | the language |
| **pandas** | tabular data — a spreadsheet you can program |
| **`math9102`** | this module's own helper package |

`math9102` is deliberately thin. It supplies the module's datasets, its house
figure style, and the reporting functions — the things you would otherwise
copy and paste every week. Everything statistical is done with **scipy**,
**statsmodels** and **pingouin** directly, because those are the tools you will
use after this module ends.

In [ ]:
import math9102 as m9
import numpy as np
import pandas as pd

m9.use_house_style()

If that cell ran without error, your environment is working.

### Why the environment is pinned

Every version of every package this module uses is fixed in
`requirements.lock`. That is not bureaucracy. An analysis that gives one answer
on your machine and another on mine is not an analysis, and the version of a
package is part of the method.

You will meet scripts that install packages at the top, inside a wrapper that
suppresses all errors. If the install fails, the script carries on and fails
later somewhere unrelated. Don't do that.

---

## Part 2 — Values, Series and DataFrames

A **variable** in Python holds a value.

In [ ]:
income = 100
taxes = 20
net_income = income - taxes
net_income

A **Series** holds many values of the same kind, and arithmetic applies to all
of them at once.

In [ ]:
age = pd.Series([28, 48, 47, 71, 22, 80, 48, 30, 31])
age * 12

In [ ]:
print(age.min(), age.max(), age.sum())

A **DataFrame** holds several Series side by side, sharing one index — one row
per case.

In [ ]:
purchase = pd.Series([20, 59, 2, 12, 22, 160, 34, 34, 29])
techsales = pd.DataFrame({"age": age, "purchase": purchase})
techsales

Selecting rows by a condition:

In [ ]:
techsales[techsales.age > 40]

Deriving a new column. Note that the categories are **mutually exclusive** and
**collectively exhaustive** — every row falls into exactly one:

In [ ]:
techsales["band"] = pd.cut(
    techsales.purchase,
    bins=[-np.inf, 20, 100, np.inf],
    labels=["cheap", "reasonable", "expensive"],
)
techsales

---

## Part 3 — Loading a dataset

**Datasets are loaded by name, never by path.**

In [ ]:
salaries = m9.load_salaries()
salaries.shape

That is a rule, not a convenience. An absolute path is a fact about one
computer — whose account, which drive, which folder — so an analysis that
hardcodes one runs on that machine and nowhere else. A dataset name is a fact
about the analysis, and it travels.

To see what is available:

In [ ]:
m9.available()

Every dataset carries its provenance — where it came from, its checksum, and
anything you need to know before using it:

In [ ]:
m9.manifest().set_index("dataset").loc[["salaries", "msleep"], ["rows", "cols", "note"]]

### What is in this one?

`salaries` records nine-month academic salaries at one US college in 2008–09.

In [ ]:
salaries.head()

In [ ]:
salaries.dtypes

In [ ]:
salaries.info()

Read `dtypes` carefully. Python's type is **not** the same thing as the
statistical level of measurement:

| Column | Python type | Level of measurement |
|---|---|---|
| `rank` | object (text) | ordinal — assistant, associate, full |
| `discipline` | object (text) | nominal — theoretical or applied |
| `sex` | object (text) | nominal |
| `yrs_since_phd` | int | ratio — a true zero, and twice as many years means twice as long |
| `yrs_service` | int | ratio |
| `salary` | int | ratio |

Python cannot tell you which column is ordinal. **That is your judgement, and
it decides which statistics are legitimate.**

---

## Part 4 — Describing a categorical variable

For a categorical variable there is only one honest summary: **how many, and
what share**.

In [ ]:
m9.frequency(salaries, "rank")

In [ ]:
m9.frequency(salaries, "discipline")

Two categorical variables together give a contingency table:

In [ ]:
m9.crosstab(salaries, "discipline", "rank")

Counts alone are hard to compare when the row totals differ. Percentages
**within rows** answer "of the people in this discipline, what share hold each
rank?":

In [ ]:
m9.crosstab(salaries, "discipline", "rank", normalize="index")

Percentages **within columns** answer a different question — "of the people at
this rank, what share are in each discipline?":

In [ ]:
m9.crosstab(salaries, "discipline", "rank", normalize="columns")

Both are correct. They answer different questions, and a table of percentages
that does not say which way it was computed is unreadable. **Always say.**

---

## Part 5 — Central tendency

Three summaries of "the typical value", and they do not agree.

In [ ]:
print(f"mean   = {salaries.salary.mean():,.0f}")
print(f"median = {salaries.salary.median():,.0f}")
print(f"mode   = {salaries.salary.mode().iloc[0]:,.0f}")

The mean is higher than the median. That is the signature of a **right-skewed**
distribution: a tail of large values pulls the mean up, and the median does not
follow.

In [ ]:
m9.histogram_with_normal(salaries.salary, xlabel="Nine-month salary (USD)");

### Which one should you report?

- **Mode** — the only one available for a nominal variable.
- **Median** — the middle value. Unaffected by how extreme the extremes are.
- **Mean** — uses every value, which is its strength and its weakness.

A worked illustration. Consider a neighbourhood of twenty people earning a
modest wage, one earning rather more, five professionals, and a film star:

In [ ]:
neighbourhood = pd.Series([10_000] * 20 + [35_000] + [150_000] * 5 + [4_465_000])

print(f"mode   = {neighbourhood.mode().iloc[0]:,.0f}")
print(f"median = {neighbourhood.median():,.0f}")
print(f"mean   = {neighbourhood.mean():,.0f}")

Nobody in that street earns anything close to the mean. It is not a wrong
calculation; it is a misleading summary, and reporting it alone would be a
choice you would have to defend.

> There is an old story about the statistician who drowned crossing a lake with
> an average depth of three feet.

---

## Part 6 — Dispersion

A measure of the centre means little without a measure of the spread.

In [ ]:
salary = salaries.salary

print(f"range    = {salary.max() - salary.min():,.0f}")
print(f"IQR      = {salary.quantile(.75) - salary.quantile(.25):,.0f}")
print(f"variance = {salary.var(ddof=1):,.0f}")
print(f"SD       = {salary.std(ddof=1):,.0f}")

**Range** is the distance from smallest to largest. It uses two observations and
ignores the other several hundred, so a single extreme value sets it.

**IQR** is the width of the middle half. It ignores the tails by construction,
which is exactly why it survives them.

**Variance** is the average squared distance from the mean. Squaring stops
positive and negative deviations cancelling — and it is preferred to taking
absolute values because the squared version has the mathematical properties the
rest of statistics is built on.

**Standard deviation** is the square root of the variance, which puts it back
into the units of the original variable. That is why it is the one you report.

### Why `ddof=1`

In [ ]:
print(f"pandas default (ddof=1): {salary.std():,.3f}")
print(f"numpy  default (ddof=0): {np.std(salary):,.3f}")

`ddof` is the divisor correction: with `ddof=1` you divide by one less than the
number of observations. Estimating the mean from the sample uses up one piece of
information, so only the rest are free to vary. **pandas defaults to `ddof=1`,
numpy defaults to `ddof=0`.** They will silently disagree. Know which you are
using.

### Reading a standard deviation

For a roughly **normal** distribution, about two thirds of observations fall
within one standard deviation of the mean, and almost all within three. Week 2
makes this exact. `salary` is not normal — it has the right tail you saw above —
so treat the rule as a sanity check, and notice how close it still lands:

In [ ]:
lower, upper = salary.mean() - salary.std(), salary.mean() + salary.std()
inside = salary.between(lower, upper).mean()

print(f"within one SD of the mean: {inside:.1%}")
print(f"interval: {lower:,.0f} to {upper:,.0f}")

Two groups can share a mean and be entirely different. Suppose two programmes
have the same mean exam mark but different standard deviations:

In [ ]:
for sd in (1.6, 4.3):
    print(f"SD {sd}: about two thirds of marks between "
          f"{60 - sd:.1f} and {60 + sd:.1f}, "
          f"almost all between {60 - 3 * sd:.1f} and {60 + 3 * sd:.1f}")

Same centre, very different student experience. Reporting only the mean would
hide it.

---

## Part 7 — The whole description at once

In [ ]:
m9.describe(salaries, ["salary", "yrs_since_phd", "yrs_service"])

`skew` and `kurtosis` describe the **shape**:

- **Skew** — positive means a tail to the right, negative a tail to the left,
  zero means symmetric.
- **Kurtosis** — how much of the variance comes from rare extreme values rather
  than from typical ones.

Treat both as descriptions, not as verdicts. We come back to them in week 3.

And split by a group:

In [ ]:
m9.describe_by(salaries, "salary", "discipline")

---

## Part 8 — Missing data, on day one

Real datasets are incomplete. The lab dataset is a good example.

In [ ]:
msleep = m9.load_msleep()
msleep.head()

In [ ]:
msleep.isna().sum()

More than half the animals have no recorded sleep cycle. That is not a defect in
the data — it is a fact about how hard that measurement is to take — but it
governs what you can say.

In [ ]:
missing = m9.missingness(msleep, ["sleep_total", "vore", "bodywt"])
print(missing.attrs["summary"])
missing

Now the same question for a different set of variables:

In [ ]:
missing_cycle = m9.missingness(msleep, ["sleep_cycle", "brainwt", "vore"])
print(missing_cycle.attrs["summary"])

**The cost of dropping incomplete rows depends entirely on which variables you
are using.** So choose your variables first, then look at the cost. Doing it the
other way round — dropping every row with any missing value anywhere in the
dataset — throws away cases for columns your analysis never touches. On a wide
dataset that can cost you a large share of the sample, most of it to variables
no model of yours will mention.

---

## What to take from this week

- A dataset is rows of cases and columns of variables. **You** decide each
  variable's level of measurement, and that decision governs everything after.
- Describe a categorical variable with counts and percentages, and say which way
  you percentaged.
- Describe a continuous variable with a centre **and** a spread, and look at its
  shape before choosing which centre.
- Never report a mean without knowing whether the distribution is skewed.
- Load data by name. Pin your environment. Look at missingness before you drop
  anything.

## Exercise

`exercise.md` gives you a dataset you have not seen and asks you to do all of
this yourself.